# Unit 7 - 00 - Prepare SR-CACO-2 (X4) pairs

[SR-CACO-2](https://github.com/sbelharbi/sr-caco-2) ships **pre-cropped, pre-registered**
low/high-resolution patch pairs, so preparation is mostly matching each
low-res patch to its high-res patch

**X4 task:** 128x128 low-res patches (folder `hr_div_4`, a *real* low-resolution confocal
scan) -> 512x512 high-res (folder `hr_div_1`).

> **Data** Loaded from a prepared ~1.1 GB zip in persistent AzureML cloudfiles
> (`~/cloudfiles/data/srcaco2_x4_cell2.zip`, CELL2 / X4 patches), extracted into a local
> working copy on first run. (Belharbi et al., NeurIPS 2024).

In [ ]:
import os, re, csv, glob, random
import numpy as np
import tifffile
from skimage.transform import resize
import matplotlib.pyplot as plt

random.seed(0)


## Configuration

The prepared CELL2 / X4 patches live in persistent cloudfiles (`SRC_ZIP`); on first run
they are extracted into a local `DATA_ROOT` working copy (reused if already present).

`SCALE = 4` is the X4 task (LR `hr_div_4`, 128px -> HR `hr_div_1`, 512px), marker `CELL2`.
(Other markers/scales would require re-creating the zip from the full dataset.)

In [ ]:
# Local working copy (extracted from the cloudfiles zip below on first run).
DATA_ROOT = "./srcaco2_data"

# Prepared SR-CACO-2 X4 / CELL2 patches, stored in persistent AzureML cloudfiles.
SRC_ZIP = os.path.expanduser("~/cloudfiles/code/data/srcaco2_x4_cell2.zip")

MARKER = "CELL2"   # the zip holds CELL2 only
SCALE  = 4         # X4: LR hr_div_4 (128px) -> HR hr_div_1 (512px)

OUT_DIR  = "./prepared"
MANIFEST = os.path.join(OUT_DIR, "pairs_x%d_%s.csv" % (SCALE, MARKER))
os.makedirs(OUT_DIR, exist_ok=True)

# Official tile-based split (by tile ID) from the SR-CACO-2 paper.
VAL_TILES  = {7, 11, 19}
TEST_TILES = {9, 10, 14, 20}

In [ ]:
import zipfile

DATA_ROOT = os.path.abspath(DATA_ROOT)
os.makedirs(DATA_ROOT, exist_ok=True)
DIV_HR = "hr_div_1"               # HR target (512x512 patches)
DIV_LR = "hr_div_%d" % SCALE      # X4 -> hr_div_4 (128px)

def find_dir(name):
    hits = [p for p in glob.glob(os.path.join(DATA_ROOT, "**", name), recursive=True) if os.path.isdir(p)]
    return hits[0] if hits else None

HR_DIR, LR_DIR = find_dir(DIV_HR), find_dir(DIV_LR)

if not (HR_DIR and LR_DIR):
    assert os.path.exists(SRC_ZIP), (
        "Prepared patches not found locally and the cloudfiles zip is missing:\n  %s\n"
        "(It should contain caco2/%s and caco2/%s for marker %s.)" % (SRC_ZIP, DIV_HR, DIV_LR, MARKER)
    )
    print("Extracting %s -> %s ..." % (SRC_ZIP, DATA_ROOT))
    with zipfile.ZipFile(SRC_ZIP) as z:
        z.extractall(DATA_ROOT)
    HR_DIR, LR_DIR = find_dir(DIV_HR), find_dir(DIV_LR)

assert HR_DIR and LR_DIR, (
    "Could not locate %s / %s under %s after extracting the zip." % (DIV_HR, DIV_LR, DATA_ROOT)
)
print("HR dir:", HR_DIR, "->", len(glob.glob(os.path.join(HR_DIR, "*%s.tif" % MARKER))), "patches")
print("LR dir:", LR_DIR, "->", len(glob.glob(os.path.join(LR_DIR, "*%s.tif" % MARKER))), "patches")

## Pair the patches

Patch filenames look like `tile_<Prefix>-<tileID>_<patchID>_<coords...>_CELL<i>.tif`
(the `<Prefix>` encodes the source scan, not the patch size). The `(tileID, patchID, cell)`
triple identifies the same field of view across resolutions, so we join HR and LR on it.

In [ ]:
# Extract (tile_id, patch_id, cell) from a patch filename.
KEY_RE = re.compile(r"-(?P<tile>\d+)_(?P<patch>\d+)_.*_CELL(?P<cell>\d+)\.tif$", re.IGNORECASE)

def index_dir(d, marker):
    out = {}
    for path in glob.glob(os.path.join(d, "*.tif")):
        m = KEY_RE.search(os.path.basename(path))
        if not m or ("CELL" + m.group("cell")) != marker:
            continue
        out[(int(m.group("tile")), int(m.group("patch")), int(m.group("cell")))] = path
    return out

hr_index = index_dir(HR_DIR, MARKER)
lr_index = index_dir(LR_DIR, MARKER)
print("HR patches (%s): %d" % (MARKER, len(hr_index)))
print("LR patches (%s): %d" % (MARKER, len(lr_index)))

keys = sorted(set(hr_index) & set(lr_index))
print("Matched pairs:", len(keys))
assert keys, "No matched pairs - check DATA_ROOT / MARKER / folder names."


In [ ]:
def split_for(tile_id):
    if tile_id in TEST_TILES:
        return "test"
    if tile_id in VAL_TILES:
        return "val"
    return "train"

rows = []
for (tile, patch, cell) in keys:
    rows.append({
        "split": split_for(tile),
        "tile_id": tile,
        "patch_id": patch,
        "cell": cell,
        "lr_path": os.path.abspath(lr_index[(tile, patch, cell)]),
        "hr_path": os.path.abspath(hr_index[(tile, patch, cell)]),
    })

with open(MANIFEST, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["split", "tile_id", "patch_id", "cell", "lr_path", "hr_path"])
    w.writeheader()
    w.writerows(rows)

from collections import Counter
counts = Counter(r["split"] for r in rows)
print("Wrote", MANIFEST)
for s in ("train", "val", "test"):
    print("  %-5s %d pairs" % (s, counts.get(s, 0)))


## Visual sanity checks

A few cheap checks before training: (1) LR/HR pairs are aligned and the HR really is
sharper, (2) intensities sit in the 8-bit `[0, 255]` range, (3) shapes/dtypes are what
we expect (LR 128x128, HR 512x512, single channel).


In [ ]:
def load_gray(path):
    img = tifffile.imread(path)
    if img.ndim == 3:               # defensive: collapse any stray channel axis
        img = img[..., 0] if img.shape[-1] <= 4 else img[0]
    return np.asarray(img)

# (1) LR (upsampled for display) vs HR for a few random training pairs
train_rows = [r for r in rows if r["split"] == "train"]
sample = random.sample(train_rows, k=min(4, len(train_rows)))

fig, axes = plt.subplots(len(sample), 2, figsize=(6, 3 * len(sample)))
axes = np.atleast_2d(axes)
for i, r in enumerate(sample):
    lr, hr = load_gray(r["lr_path"]), load_gray(r["hr_path"])
    lr_up = resize(lr, hr.shape, order=3, preserve_range=True, anti_aliasing=False)
    axes[i, 0].imshow(lr_up, cmap="gray"); axes[i, 0].set_title("LR %s (up to HR size)" % str(lr.shape))
    axes[i, 1].imshow(hr,    cmap="gray"); axes[i, 1].set_title("HR %s" % str(hr.shape))
    for a in axes[i]:
        a.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# (2) intensity histograms (LR vs HR) on one example
lr, hr = load_gray(sample[0]["lr_path"]), load_gray(sample[0]["hr_path"])
plt.figure(figsize=(6, 3))
plt.hist(lr.ravel(), bins=50, alpha=0.6, label="LR")
plt.hist(hr.ravel(), bins=50, alpha=0.6, label="HR")
plt.xlabel("pixel value"); plt.ylabel("count"); plt.legend()
plt.title("Intensity distribution (expect 8-bit range 0-255)")
plt.show()

# (3) shapes / dtype / range
print("LR:", lr.shape, lr.dtype, "min/max", lr.min(), lr.max())
print("HR:", hr.shape, hr.dtype, "min/max", hr.min(), hr.max())


That's it for prep - the manifest CSV now lists every aligned `(LR, HR)` pair and its
split. Continue to **`01_Train.ipynb`**.
